# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AshantiVilladiego/FlyRankAI-Internship-Starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### 1. The Data Contract

1. **Unit of analysis (The Grain):** One row represents the daily performance of one specific piece of content for one client (`report_date` × `client_hash_id` × `content_hash_id`).
2. **Tables used:** `fact_content_daily_performance` (specifically the mid-panel partition `month=2026-03`).
3. **Time window:** March 1, 2026, through March 31, 2026.
4. **Label / Proxy:** I will predict whether the content generated any search traffic tomorrow (`target_clicked_tomorrow` = next day's `gsc_clicks > 0`).
5. **Deliberate Exclusion:** I am excluding any rows where `ga4_data_available` is `FALSE`. The documentation warns that rows prior to a client's GA4 setup are zero-filled, meaning a `0` is a missing value, not a lack of engagement.
6. **Output:** This model acts as a **decision-support** tool by providing a **directional** signal of whether search traffic will be **observed** tomorrow.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

I am predicting `target_clicked_tomorrow` (whether a click is **observed** on the *next* day).

1. **`report_day_of_week`**: Knowable at the decision moment because it is a fixed calendar property of the current row's date.
2. **`is_weekend`**: Knowable at the decision moment because it relies solely on the current calendar date.
3. **`today_impressions`**: Knowable at the decision moment (midnight) because the current day's **measured** Search Console impressions have finished accruing before we predict tomorrow.
4. **`today_avg_position`**: Knowable at the decision moment because today's **measured** ranking data is finalized before tomorrow begins.
5. **`today_ga4_sessions`**: Knowable at the decision moment because today's **measured** Google Analytics sessions are recorded prior to the target prediction window.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*



In [7]:
import pandas as pd
import os
from google.colab import userdata

# 1. Load the specific mid-panel month partition
hf_token = userdata.get('hf_token')
df_mar = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03",
    storage_options={"token": hf_token}
)

print("--- FACT 1: PROVE THE GRAIN ---")
# Using the correct hash_id columns
max_rows_per_group = df_mar.groupby(['report_date', 'client_hash_id', 'content_hash_id']).size().max()
print(f"Max rows per report_date x client x content: {max_rows_per_group}")
assert max_rows_per_group == 1, "Grain violation detected!"

print("\n--- FACT 2: ROW COUNT & DATE SPAN ---")
row_count = len(df_mar)
min_date = df_mar['report_date'].min()
max_date = df_mar['report_date'].max()
print(f"Total Rows: {row_count:,}")
print(f"Date Span: {min_date} to {max_date}")

print("\n--- FACT 3: AVAILABILITY (IS TRUE) ---")
# Enforcing the panel warning rule: filter out zero-filled GA4 data
df_clean = df_mar[df_mar['ga4_data_available'] == True].copy()
surviving_rows = len(df_clean)
print(f"Rows surviving 'ga4_data_available == True': {surviving_rows:,} ({(surviving_rows/row_count)*100:.1f}%)")

--- FACT 1: PROVE THE GRAIN ---
Max rows per report_date x client x content: 1

--- FACT 2: ROW COUNT & DATE SPAN ---
Total Rows: 9,841,378
Date Span: 2026-03-01 to 2026-03-31

--- FACT 3: AVAILABILITY (IS TRUE) ---
Rows surviving 'ga4_data_available == True': 413,966 (4.2%)


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

# --- PREPARE DATA & SHIFT TARGET ---
# Sort to ensure time-series integrity before shifting
df_clean = df_clean.sort_values(['client_hash_id', 'content_hash_id', 'report_date'])

# Create the honest target: Did they get a click TOMORROW?
df_clean['target_clicked_tomorrow'] = df_clean.groupby(['client_hash_id', 'content_hash_id'])['gsc_clicks'].shift(-1) > 0

# Drop the last day of the month since we can't know "tomorrow" for it
df_model = df_clean.dropna(subset=['target_clicked_tomorrow']).copy()

# --- BUILD THE 5 HONEST FEATURES ---
# FIX: Convert the date to a Pandas datetime object first so .dt works
df_model['report_date_pd'] = pd.to_datetime(df_model['report_date'])
df_model['report_day_of_week'] = df_model['report_date_pd'].dt.dayofweek
df_model['is_weekend'] = df_model['report_day_of_week'].isin([5, 6]).astype(int)
df_model['today_impressions'] = df_model['gsc_impressions'].fillna(0)

# Handling the gotcha: avg_position = 0 means "no data", not rank 0. Replace 0 with a high penalty rank
df_model['today_avg_position'] = df_model['gsc_avg_position'].replace(0, 100)
df_model['today_ga4_sessions'] = df_model['ga4_sessions'].fillna(0)

honest_features = ['report_day_of_week', 'is_weekend', 'today_impressions', 'today_avg_position', 'today_ga4_sessions']

# --- ADD THE TRAP (LEAKAGE) ---
# Trap: We accidentally include TOMORROW'S impressions in the feature set
df_model['TRAP_tomorrow_impressions'] = df_model.groupby(['client_hash_id', 'content_hash_id'])['gsc_impressions'].shift(-1)
# Drop NaNs created by the trap shift just for the experiment
df_exp = df_model.dropna(subset=['TRAP_tomorrow_impressions']).copy()

X_leak = df_exp[honest_features + ['TRAP_tomorrow_impressions']]
y = df_exp['target_clicked_tomorrow']

# --- TRAIN WITH LEAKAGE ---
X_train_L, X_test_L, y_train_L, y_test_L = train_test_split(X_leak, y, test_size=0.2, random_state=42)
clf_leak = RandomForestClassifier(max_depth=5, random_state=42).fit(X_train_L, y_train_L)
leak_score = accuracy_score(y_test_L, clf_leak.predict(X_test_L))
print(f"❌ Score WITH the Trap (Target Leakage): {leak_score:.4f} (Suspiciously high!)")

# --- TRAIN HONESTLY ---
X_honest = df_exp[honest_features]
X_train_H, X_test_H, y_train_H, y_test_H = train_test_split(X_honest, y, test_size=0.2, random_state=42)
clf_honest = RandomForestClassifier(max_depth=5, random_state=42).fit(X_train_H, y_train_H)
honest_score = accuracy_score(y_test_H, clf_honest.predict(X_test_H))
print(f"✅ Honest Score (Trap Removed): {honest_score:.4f}")

❌ Score WITH the Trap (Target Leakage): 0.7285 (Suspiciously high!)
✅ Honest Score (Trap Removed): 0.7181


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Limitation: This slice relies on a single mid-panel month (March 2026). Because history depth differs wildly per client, taking a rigid calendar slice means we capture mature content for some clients, but only the very beginning of the lifecycle for clients who onboarded in late February. This introduces maturity bias into the model.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.